In [1]:
import os
import gc
import json
import glob
import torch
import resource
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import multiprocessing
import scipy.stats as stats
from sklearn.metrics import roc_curve, auc, roc_auc_score

# ==============================================================================
# 0. CONFIGURACIÓN Y UTILIDADES VECTORIZADAS
# ==============================================================================
def configure_system_resources(max_ram_gb=100, target_cores=14):
    limit_bytes = int(max_ram_gb * 1024**3)
    try:
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        resource.setrlimit(resource.RLIMIT_AS, (limit_bytes, hard))
    except Exception as e: print(f"[!] RAM Limit error: {e}")

    available_cores = multiprocessing.cpu_count() if 'multiprocessing' in globals() else 14
    use_cores = min(target_cores, available_cores)
    torch.set_num_threads(use_cores)
    os.environ["OMP_NUM_THREADS"] = str(use_cores)
    return use_cores

def fast_parse_vectorized(df_col, max_p):
    df_col = df_col.str.replace('"', '', regex=False)
    expanded = df_col.str.split(';', expand=True).iloc[:, :max_p].astype(np.float32).fillna(0.0)
    if expanded.shape[1] < max_p:
        padding = np.zeros((len(expanded), max_p - expanded.shape[1]), dtype=np.float32)
        return np.hstack([expanded.values, padding])
    return expanded.values

def calculate_jet_axis_vectorized(pt, eta, phi):
    sum_pt = np.sum(pt, axis=1, keepdims=True) + 1e-8
    w_eta = np.sum(pt * eta, axis=1, keepdims=True) / sum_pt
    w_x = np.sum(pt * np.cos(phi), axis=1, keepdims=True)
    w_y = np.sum(pt * np.sin(phi), axis=1, keepdims=True)
    w_phi = np.arctan2(w_y, w_x)
    return w_eta, w_phi

# ==============================================================================
# 1. PREPROCESADOR
# ==============================================================================
class JetPreprocessorCSV:
    def __init__(self, max_particles=30):
        self.max_particles = max_particles
        self.mean, self.std = None, None

    def fit(self, file_pattern):
        print("[*] Calculando estadísticas vectorizadas...")
        files = glob.glob(file_pattern)
        if not files: return
        
        df = pd.read_csv(files[0], usecols=['PF_Pt', 'PF_Eta', 'PF_Phi'], nrows=100000)
        pt = fast_parse_vectorized(df['PF_Pt'], self.max_particles)
        eta = fast_parse_vectorized(df['PF_Eta'], self.max_particles)
        phi = fast_parse_vectorized(df['PF_Phi'], self.max_particles)
        
        j_eta, j_phi = calculate_jet_axis_vectorized(pt, eta, phi)
        logpt = np.log1p(pt)
        deta = (eta - j_eta) * (pt > 0)
        dphi = ((phi - j_phi + np.pi) % (2 * np.pi) - np.pi) * (pt > 0)
        
        mask = pt > 0
        self.mean = np.array([logpt[mask].mean(), deta[mask].mean(), dphi[mask].mean()], dtype=np.float32)
        self.std = np.array([logpt[mask].std(), deta[mask].std(), dphi[mask].std()], dtype=np.float32)
        print(f"   Mean: {self.mean} | Std: {self.std}")

    def save(self, path="scaler_csv.json"):
        with open(path, "w") as f:
            json.dump({"mean": self.mean.tolist(), "std": self.std.tolist()}, f)

    def load(self, path="scaler_csv.json"):
        with open(path, "r") as f:
            d = json.load(f)
            self.mean, self.std = np.array(d["mean"], dtype=np.float32), np.array(d["std"], dtype=np.float32)

# ==============================================================================
# 2. DATASET CON CACHÉ BINARIO
# ==============================================================================
class JetDatasetOptimized(Dataset):
    def __init__(self, file_path, preprocessor, max_particles=30, force_reprocess=False, max_rows=None):
        self.cache_path = file_path.replace(".csv", "_cache.pt")
        
        if force_reprocess and os.path.exists(self.cache_path):
            os.remove(self.cache_path)
            print(f"[!] Caché antiguo eliminado: {os.path.basename(self.cache_path)}")
        
        if os.path.exists(self.cache_path) and not force_reprocess:
            print(f"[>] Cargando caché actualizado: {os.path.basename(self.cache_path)}")
            self.data, self.masks, self.physics = torch.load(self.cache_path)
            if max_rows is not None and len(self.data) > max_rows:
                self.data = self.data[:max_rows]
                self.masks = self.masks[:max_rows]
                self.physics = self.physics[:max_rows]
        else:
            print(f"[>] Procesando vectorialmente y extrayendo física: {os.path.basename(file_path)}")
            df = pd.read_csv(file_path, nrows=max_rows) 
            
            pt = fast_parse_vectorized(df['PF_Pt'], max_particles)
            eta = fast_parse_vectorized(df['PF_Eta'], max_particles)
            phi = fast_parse_vectorized(df['PF_Phi'], max_particles)
            
            px, py = pt * np.cos(phi), pt * np.sin(phi)
            pz, e = pt * np.sinh(eta), pt * np.cosh(eta)
            
            jet_px, jet_py = np.sum(px, axis=1), np.sum(py, axis=1) 
            jet_pz, jet_e = np.sum(pz, axis=1), np.sum(e, axis=1)
            
            jet_mass2 = jet_e**2 - jet_px**2 - jet_py**2 - jet_pz**2
            jet_mass = np.sqrt(np.maximum(0, jet_mass2))
            jet_pt = np.sqrt(jet_px**2 + jet_py**2)

            tau1 = df['Tau1'].values.astype(np.float32) + 1e-8
            tau2 = df['Tau2'].values.astype(np.float32)
            tau21 = tau2 / tau1
            SoftMass = df['SDMass'].values.astype(np.float32)
            
            if 'is_signal' in df.columns:
                is_signal = df['is_signal'].values.astype(np.float32)
            else:
                is_signal = np.zeros(len(df), dtype=np.float32)
            
            n_constituents = np.sum(pt > 0, axis=1)
            # Índices: 0:Mass, 1:Pt, 2:N, 3:Tau21, 4:SoftMass, 5:is_signal
            physics_vars = np.stack([jet_mass, jet_pt, n_constituents, tau21, SoftMass, is_signal], axis=1).astype(np.float32)
            
            j_eta, j_phi = calculate_jet_axis_vectorized(pt, eta, phi)
            logpt = np.log1p(pt)
            deta = (eta - j_eta)
            dphi = (phi - j_phi + np.pi) % (2 * np.pi) - np.pi
            
            features = np.stack([logpt, deta, dphi], axis=2).astype(np.float32)
            features = (features - preprocessor.mean) / (preprocessor.std + 1e-8)
            masks = (pt > 0).astype(np.float32)
            features = features * np.expand_dims(masks, axis=2) 
            
            self.data = torch.from_numpy(features)
            self.masks = torch.from_numpy(masks)
            self.physics = torch.from_numpy(physics_vars)
            
            torch.save((self.data, self.masks, self.physics), self.cache_path)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.masks[idx], self.physics[idx]

# ==============================================================================
# 3. MODELO, DISTANCIA Y PENALIZACIÓN DISCO
# ==============================================================================
def chamfer_distance_fast(p1, p2, mask1, mask2, reduction='mean'):
    dist = torch.cdist(p1, p2, p=2)**2
    m1, m2 = mask1.unsqueeze(2), mask2.unsqueeze(1)
    dist = dist + (1 - m1) * 1e8 + (1 - m2) * 1e8

    loss_12 = torch.min(dist, dim=2)[0] * mask1 
    loss_21 = torch.min(dist, dim=1)[0] * mask2 

    per_jet_loss = loss_12.sum(dim=1) + loss_21.sum(dim=1)
    n_particles = mask1.sum(dim=1) + 1e-8
    per_jet_loss = per_jet_loss / n_particles

    if reduction == 'none': return per_jet_loss
    return per_jet_loss.mean()

def calc_distance_correlation(x, y):
    x, y = x.view(-1, 1), y.view(-1, 1)
    A = torch.abs(x - x.t())
    B = torch.abs(y - y.t())
    
    A_mean_row, A_mean_col, A_mean_all = A.mean(dim=0, keepdim=True), A.mean(dim=1, keepdim=True), A.mean()
    A_c = A - A_mean_row - A_mean_col + A_mean_all
    
    B_mean_row, B_mean_col, B_mean_all = B.mean(dim=0, keepdim=True), B.mean(dim=1, keepdim=True), B.mean()
    B_c = B - B_mean_row - B_mean_col + B_mean_all
    
    dcov2_xy = (A_c * B_c).mean()
    dcov2_xx = (A_c * A_c).mean()
    dcov2_yy = (B_c * B_c).mean()
    return dcov2_xy / (torch.sqrt(dcov2_xx * dcov2_yy) + 1e-8)

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=16):
        super().__init__()
        self.k = k
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch * 2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2)
        )
    def forward(self, x):
        b, c, n = x.shape
        inner = -2 * torch.matmul(x.transpose(2, 1), x)
        xx = torch.sum(x**2, dim=1, keepdim=True)
        dist = -xx - inner - xx.transpose(2, 1)
        idx = dist.topk(k=self.k, dim=-1)[1]
        
        idx_base = torch.arange(0, b, device=x.device).view(-1, 1, 1) * n
        idx = (idx + idx_base).view(-1)
        x_flat = x.transpose(2, 1).contiguous().view(b * n, c)
        neighbors = x_flat[idx, :].view(b, n, self.k, c).permute(0, 3, 1, 2)
        x_cent = x.unsqueeze(3).repeat(1, 1, 1, self.k)
        out = self.conv(torch.cat([x_cent, neighbors - x_cent], dim=1)).mean(dim=3)
        return out + x if c == out.shape[1] else out

class ParticleNetAE(nn.Module):
    def __init__(self, input_dim=3, n_part=30, latent=16):
        super().__init__()
        self.n_part = n_part
        self.enc = nn.Sequential(EdgeConv(input_dim, 64), EdgeConv(64, 128), EdgeConv(128, 256))
        
        # [MEJORA 1]: Cambio a MaxPool para retener features de subjets (W prongs)
        self.pool = nn.AdaptiveMaxPool1d(1) 
        
        self.fc_z = nn.Linear(256, latent)
        
        # [MEJORA 2]: Añadidos BatchNorms al decoder para estabilizar el entrenamiento
        self.dec = nn.Sequential(
            nn.Linear(latent, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Linear(512, n_part * input_dim)
        )
        
    def forward(self, x):
        z = self.fc_z(self.pool(self.enc(x.transpose(1, 2))).squeeze(-1))
        return self.dec(z).view(-1, self.n_part, 3)
        
    def encode(self, x):
        return self.fc_z(self.pool(self.enc(x.transpose(1, 2))).squeeze(-1))

# ==============================================================================
# 4. EVALUACIONES Y SCORE HÍBRIDO
# ==============================================================================

# [NUEVO]: Función robusta para calcular un score de anomalía que incorpora física
def compute_hybrid_anomaly_score(model, loader, device, alpha=1.0, beta=0.1, gamma=1.5, delta=1.0):
    """
    Combina Chamfer Distance con variables físicas (Tau21 y SoftDrop Mass)
    Altos scores = Anomalía (Señal WW)
    """
    model.eval()
    all_scores, all_labels = [], []

    with torch.no_grad():
        for x, m, phys in loader:
            x, m = x.to(device), m.to(device)
            phys = phys.to(device)

            recon = model(x)
            
            # 1. Reconstrucción (Chamfer). Anómalo reconstruye peor -> score alto.
            chamfer = chamfer_distance_fast(x, recon, m, m, reduction='none')

            # 2. Espacio latente. Distancia al origen.
            z = model.encode(x)
            z_norm = torch.norm(z, dim=1)

            # 3. Física del Jet (Índice 3: Tau21, Índice 4: SoftMass, Índice 5: is_signal)
            tau21 = phys[:, 3]
            soft_mass = phys[:, 4]
            
            # Queremos premiar Ws: bajo tau21 (1 - tau21 es alto)
            tau21_score = torch.clamp(1.0 - tau21, min=0.0) 
            
            # Queremos premiar masas cerca a 80 GeV (Bosón W)
            mass_score = torch.exp(-0.5 * ((soft_mass - 80.0) / 15.0)**2) 

            # COMPOSICIÓN DEL SCORE
            score = (alpha * chamfer) + (beta * z_norm) + (gamma * tau21_score) + (delta * mass_score)

            all_scores.extend(score.cpu().numpy())
            all_labels.extend(phys[:, 5].cpu().numpy()) 

    return np.array(all_scores), np.array(all_labels)

def diagnosticar_y_evaluar_roc(model, loader, device):
    print("\n" + "=" * 50)
    print("[*] Generando Score Híbrido y Evaluando Curva ROC...")
    
    # [MEJORA 3]: Usamos el score híbrido en lugar de solo Chamfer
    scores, labels = compute_hybrid_anomaly_score(model, loader, device, alpha=1.0, beta=0.1, gamma=2.0, delta=1.0)

    print(f"Total jets evaluados : {len(labels)}")
    if len(np.unique(labels)) < 2 or labels.sum() == 0:
        print("[!] CRÍTICO: No hay suficientes eventos de señal (is_signal=1) en el test set para calcular ROC.")
        print("=" * 50)
        return

    print(f"Jets de señal (WW) : {labels.sum():.0f}  ({labels.mean()*100:.1f}%)")
    print(f"Jets QCD (Fondo)   : {(1-labels).sum():.0f}")
    print(f"Score QCD   — media: {scores[labels==0].mean():.4f}  std: {scores[labels==0].std():.4f}")
    print(f"Score Señal — media: {scores[labels==1].mean():.4f}  std: {scores[labels==1].std():.4f}")

    # Calcular AUC Normal
    auc_normal = roc_auc_score(labels, scores)
    print(f"\n[+] AUC SCORE HÍBRIDO : {auc_normal:.4f}")

    # Gráficos
    fpr, tpr, _ = roc_curve(labels, scores)
    bkg_rejection = 1.0 / (fpr + 1e-10)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Curva ROC Tradicional
    ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AE + Física (AUC = {auc_normal:.3f})')
    ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax1.set_xlim([0.0, 1.0])
    ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('Aceptación de QCD (FPR)')
    ax1.set_ylabel('Eficiencia de Señal WW (TPR)')
    ax1.set_title('Curva ROC')
    ax1.legend(loc="lower right")
    ax1.grid(True, alpha=0.3)

    # Curva HEP (Rejection)
    ax2.plot(tpr, bkg_rejection, color='blue', lw=2, label=f'AUC = {auc_normal:.3f}')
    ax2.set_yscale('log')
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([1, max(bkg_rejection) * 1.5]) 
    ax2.set_xlabel('Eficiencia de Señal WW (TPR)')
    ax2.set_ylabel('Rechazo de Fondo QCD (1 / FPR)')
    ax2.set_title('Rendimiento del Tagger (Escala Log)')
    ax2.legend(loc="upper right")
    ax2.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.savefig('grafica_roc_hibrida.png', dpi=300)
    plt.close()
    print("[*] Gráficos ROC guardados en 'grafica_roc_hibrida.png'.")
    print("=" * 50)


# ==============================================================================
# 5. MAIN Y BUCLE DE ENTRENAMIENTO
# ==============================================================================
def main():
    configure_system_resources()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    MAX_PARTICLES = 30
    BATCH_SIZE = 256
    TRAIN_PATTERN = "file_0.csv" 
    TEST_PATTERN = "mixed_signal_qcd.csv" # <--- Asegúrate que este archivo exista
    
    prep = JetPreprocessorCSV(MAX_PARTICLES)
    if os.path.exists("scaler_csv.json"): prep.load()
    else: 
        prep.fit(TRAIN_PATTERN)
        prep.save()

    # 1. Preparar datos de Entrenamiento / Validación (Puro QCD)
    files = glob.glob(TRAIN_PATTERN)[:1]
    datasets = [JetDatasetOptimized(f, prep, MAX_PARTICLES, force_reprocess=True, max_rows=30000) for f in files]
    full_dataset = ConcatDataset(datasets)

    train_size = int(0.9 * len(full_dataset))
    train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, len(full_dataset)-train_size])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=4)

    # 2. Preparar datos de Test
    if os.path.exists(TEST_PATTERN):
        print(f"[*] Cargando dataset mixto de prueba: {TEST_PATTERN}")
        test_ds = JetDatasetOptimized(TEST_PATTERN, prep, MAX_PARTICLES, force_reprocess=True)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=4)
    else:
        print(f"[!] Archivo mixto '{TEST_PATTERN}' no encontrado. Se omitirá la evaluación final.")
        test_loader = None

    model = ParticleNetAE(n_part=MAX_PARTICLES).to(device)
    
    if hasattr(torch, 'compile'):
        model = torch.compile(model)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    train_losses, val_losses, train_disco_vals = [], [], []
    num_epochs = 60 
    
    # [MEJORA 4]: Lambda de DisCo reducido drásticamente.
    # Decorrelacionar demasiado destruye el AE. Queremos un empujón suave, no dominar la loss.
    LAMBDA_DISCO = 0.05 

    for epoch in range(num_epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        running_train_loss, running_disco_val = 0.0, 0.0

        for x, m, phys in pbar:    
            x, m, phys = x.to(device), m.to(device), phys.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            recon = model(x)
            per_jet_recon_loss = chamfer_distance_fast(x, recon, m, m, reduction='none')
            mean_recon_loss = per_jet_recon_loss.mean()
            masses = phys[:, 4] 
            
            disco_penalty = calc_distance_correlation(per_jet_recon_loss, masses)
            total_loss = mean_recon_loss + (LAMBDA_DISCO * disco_penalty)
            
            total_loss.backward()
            optimizer.step()
            
            running_train_loss += mean_recon_loss.item()
            running_disco_val += disco_penalty.item()
            pbar.set_postfix(Recon=f"{mean_recon_loss.item():.4f}", DisCo=f"{disco_penalty.item():.4f}")
            
        avg_train_loss = running_train_loss / len(train_loader)
        avg_disco_val = running_disco_val / len(train_loader)
        train_losses.append(avg_train_loss), train_disco_vals.append(avg_disco_val)

        # Validación
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for x, m, _ in val_loader: 
                x, m = x.to(device), m.to(device)
                recon = model(x)
                loss = chamfer_distance_fast(x, recon, m, m, reduction='mean')
                running_val_loss += loss.item()
                
        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        print(f"[*] Fin Epoch {epoch+1} | Val Recon: {avg_val_loss:.4f} | Train DisCo: {avg_disco_val:.4f}")

    # --- EVALUACIÓN FINAL DE LA CURVA ROC ---
    if test_loader is not None:
        diagnosticar_y_evaluar_roc(model, test_loader, device)

if __name__ == "__main__":
    main()

[!] Caché antiguo eliminado: file_0_cache.pt
[>] Procesando vectorialmente y extrayendo física: file_0.csv
[*] Cargando dataset mixto de prueba: mixed_signal_qcd.csv
[!] Caché antiguo eliminado: mixed_signal_qcd_cache.pt
[>] Procesando vectorialmente y extrayendo física: mixed_signal_qcd.csv


Epoch 1/60 [Train]: 100%|███████████████████████████████████████████████████████████| 106/106 [00:24<00:00,  4.38it/s, DisCo=0.0270, Recon=0.3409]


[*] Fin Epoch 1 | Val Recon: 0.3292 | Train DisCo: 0.0379


Epoch 2/60 [Train]: 100%|███████████████████████████████████████████████████████████| 106/106 [00:04<00:00, 25.95it/s, DisCo=0.0636, Recon=0.3009]


[*] Fin Epoch 2 | Val Recon: 0.3153 | Train DisCo: 0.0335


...

[*] Fin Epoch 58 | Val Recon: 0.2566 | Train DisCo: 0.0295


Epoch 59/60 [Train]: 100%|██████████████████████████████████████████████████████████| 106/106 [00:04<00:00, 25.72it/s, DisCo=0.0591, Recon=0.2386]


[*] Fin Epoch 59 | Val Recon: 0.2593 | Train DisCo: 0.0294


Epoch 60/60 [Train]: 100%|██████████████████████████████████████████████████████████| 106/106 [00:04<00:00, 24.85it/s, DisCo=0.0536, Recon=0.2171]


[*] Fin Epoch 60 | Val Recon: 0.2673 | Train DisCo: 0.0290

[*] Generando Score Híbrido y Evaluando Curva ROC...
Total jets evaluados : 425717
Jets de señal (WW) : 109085  (25.6%)
Jets QCD (Fondo)   : 316632
Score QCD   — media: 2.1502  std: 0.5468
Score Señal — media: 3.1696  std: 0.6357

[+] AUC SCORE HÍBRIDO : 0.8803
[*] Gráficos ROC guardados en 'grafica_roc_hibrida.png'.
